# Before to start
Put this notebook file in the submitted Google Drive directory.

Edit **Section 1**, i.e., the first part of this notebook file implementing:

1.  **Section 1.1**: the loading of your best model(s):

- a. Create the network(s)
- b. Load the best saved model(s) using a relative path (e.g., `"./model.pth"` or `"./model/best.pth"`)

2.  **Section 1.2**: the predict function of your model(s):

- a. Pre-process the input batch of data
- b. Perform the forward using the network(s)
- c. Post-process the output of the network(s)

DO NOT EDIT **Section 2** since it replicates the exact evaluation code we run on the private test set.

To test your model you have to:

- create a directory named `"eval"` containing the test images
- run all cells

# Section 1: YOUR CODE

## 0. Mount Google Drive (only if checkpoint is on Drive)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# If your test images are on Drive, uncomment and set the path:
# import os
# os.chdir('/content/drive/MyDrive/path/to/your/project/folder')

## Section 1.1: IMPLEMENT HERE THE FUNCTION TO LOAD YOUR MODEL

**IMPORTANT: load the trained weights of your model here!**

We use ResNet18 pretrained on ImageNet, fine-tuned with two-phase training, class-weighted cross-entropy, label smoothing 0.1, and WeightedRandomSampler.

Checkpoint: `resnet18_exnovo_v322_best.pth` (v3.2.2, BA 0.9750).

In [ ]:
import torch
import torch.nn as nn
from torchvision import models

NUM_CLASSES = 8
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

def load_model():
    # Create ResNet18 with same architecture as training
    model = models.resnet18(weights=None)
    model.fc = nn.Sequential(
        nn.Dropout(p=0.25),
        nn.Linear(model.fc.in_features, NUM_CLASSES),
    )

    # Load trained weights from Google Drive checkpoint
    checkpoint_path = '/content/drive/MyDrive/ProjectML_checkpoints/resnet18_exnovo_v322_20260702-1845/resnet18_exnovo_v322_best.pth'
    checkpoint = torch.load(checkpoint_path, map_location=DEVICE, weights_only=False)
    model.load_state_dict(checkpoint['model_state_dict'])

    model.to(DEVICE)
    model.eval()

    # Log GPU memory after loading
    if DEVICE.type == 'cuda':
        print(f'Model loaded. GPU memory: {torch.cuda.memory_allocated()/1024**3:.3f} GB')

    return model

print('load_model() ready.')

### (Optional) Reject Option loading

If you want to use the reject option (Vento et al.), the thresholds below were calibrated on val_cal via coverage>=95% criterion. Uncomment to use.

In [ ]:
# Reject thresholds from v3.2.2 (calibrated on val_cal, coverage >= 95%):
#   psi_a (softmax confidence): tau = 0.4256
#   psi_b (top-2 margin):       tau = 0.1845
#
# To use reject in predict(), set REJECT_ENABLED = True
REJECT_ENABLED = False
REJECT_TAU = 0.1845        # psi_b threshold
REJECT_LABEL = 255          # label for rejected samples (outside 0-7)

print(f'Reject option: {"ENABLED" if REJECT_ENABLED else "DISABLED"}')

## Section 1.2: IMPLEMENT HERE YOUR PREDICT FUNCTION

Input: `X` is a `uint8` numpy array of shape `(batch_size, rows, cols, 3)`

Output: `uint8` numpy array of shape `(batch_size, 1)`

Preprocessing pipeline (same as training `val_transform`):
1. Convert uint8 [0,255] -> float32 [0,1]
2. Resize to 256, CenterCrop to 224
3. Normalize with ImageNet mean/std
4. Forward pass -> softmax -> argmax

In [ ]:
import numpy as np
import torch.nn.functional as F

# ImageNet normalization constants
MEAN = torch.tensor([0.485, 0.456, 0.406], device=DEVICE).view(1, 3, 1, 1)
STD  = torch.tensor([0.229, 0.224, 0.225], device=DEVICE).view(1, 3, 1, 1)

def predict(model, X):
    """
    X: numpy uint8 array of shape (batch_size, rows, cols, 3)
    Returns: numpy uint8 array of shape (batch_size, 1)
    """
    model.eval()

    with torch.no_grad():
        # 1. Convert numpy (B,H,W,C) uint8 -> torch (B,C,H,W) float32 [0,1]
        x = torch.from_numpy(X).permute(0, 3, 1, 2).float().to(DEVICE) / 255.0

        # 2. Resize to 256 and CenterCrop to 224
        x = F.interpolate(x, size=(256, 256), mode='bilinear', align_corners=False)
        # Center crop: remove 16 pixels from each side
        x = x[:, :, 16:240, 16:240]

        # 3. Normalize with ImageNet stats
        x = (x - MEAN) / STD

        # 4. Forward pass
        logits = model(x)
        probs = F.softmax(logits, dim=1)

        # 5. Predictions: argmax (or reject if enabled)
        if REJECT_ENABLED:
            # psi_b: top-2 margin (Vento et al.)
            sorted_probs, _ = probs.sort(dim=1, descending=True)
            margin = sorted_probs[:, 0] - sorted_probs[:, 1]
            reject_mask = margin < REJECT_TAU
            preds = logits.argmax(dim=1)
            preds[reject_mask] = REJECT_LABEL  # uncertain -> reject
            Y = preds.unsqueeze(1).cpu().numpy().astype(np.uint8)
        else:
            Y = logits.argmax(dim=1).unsqueeze(1).cpu().numpy().astype(np.uint8)

    return Y

print('predict() ready.')

Run this cell to verify that the produced output is well formatted

In [ ]:
import numpy as np
model = load_model()
X = np.random.randint(0, 255, size=(2, 256, 256, 3), dtype=np.uint8)
y = predict(model, X)
assert y.shape == (X.shape[0], 1) and y.dtype == np.uint8, "Verify your model loading or predict function."
valid_labels = set(range(NUM_CLASSES))
if REJECT_ENABLED:
    valid_labels.add(REJECT_LABEL)
assert all(lbl in valid_labels for lbl in y.flatten()), f"Invalid label detected. Valid: {valid_labels}"
print(f'Passed: output shape={y.shape}, dtype={y.dtype}, labels={y.flatten()}')
print(f'GPU memory: {torch.cuda.memory_allocated()/1024**3:.3f} GB (limit: 4 GB)')

# Before to run the code

1.  eventually change the current working directory using the `os.chdir` function (DO IT IN THE EMPTY CELL BELOW THIS ONE)
2.  create a directory named `"./eval"` in the current working directory
3.  verify that the `"./eval"` directory contains the image files for the test

Then, run all the cells.

In [ ]:
# If needed, change working directory here.
# Example: import os; os.chdir('/content/drive/MyDrive/your_project_folder')

# Section 2: Test code (DO NOT MODIFY THE CODE BELOW!)
This is exactly the code we run for the final test.

Just run the code to verify that it works. This is the exact code we run for the final evaluation on the private test set.

In [ ]:
# from google.colab import drive; drive.mount('/content/drive');
# import os; os.chdir("/content/drive/My Drive/Didattica/ML/exam_2025_2026/project_work_Ing_Inf/evaluation")

In [ ]:
import os
from glob import glob

test_dir = "./eval/"
assert os.path.isdir(test_dir), "The evaluation directory does not exist. Create it, put some images in it and run again this cell."

samples = [sample for sample in glob(test_dir + '/**', recursive=True) if os.path.isfile(sample)]
assert len(samples) > 0, "The evaluation directory is empty. Put some images in it and run again this cell."
print(f"Found {len(samples)} samples in {test_dir}")

In [ ]:
import os
from PIL import Image
import numpy as np
from tqdm import tqdm

# Run YOUR LOAD_MODEL FUNCTION
model = load_model()

# Main loop
verbose = True

PREDICTIONs = np.zeros((len(samples), )) * np.nan
for i, img_path in tqdm(enumerate(samples), desc="Processing samples"):
  try:
    # Open images
    rgb_image = Image.open(img_path)
    rgb_array = np.asarray(rgb_image)[None, ...]
    if verbose:
      print(f"\n  Loaded {img_path}. The input batch has shape {rgb_array.shape} and dtype {rgb_array.dtype}")

    # Run YOUR PREDICT FUNCTION
    predicted_labels_array = predict(model, rgb_array).squeeze()
    if verbose:
      print(f"  Predicted label {predicted_labels_array}")

    PREDICTIONs[i] = predicted_labels_array

  except FileNotFoundError:
    print(f"  Error: Could not find image file {img_path}")
  except Exception as e:
    print(f"  Error processing image {img_path}: {e}")

PREDICTIONs = PREDICTIONs.astype(np.uint8)
print(f"Predictions: {PREDICTIONs}")
np.save("predictions.npy", PREDICTIONs)